# Neurodesk Evidence-Informed Workflow Notebook

This notebook has been rewritten to follow the workflow structure in the proposal and thesis slides.

It is organized into five connected stages:

1. **Stage 1: Structural MRI feature inputs**
2. **Stage 2: Literature-informed evidence framework**
3. **Stage 3: Optional multimodal extension**
4. **Stage 4: Patient-to-evidence comparison**
5. **Stage 5: Structured interpretive reporting**

The core idea is to combine **structural MRI outputs** with **literature evidence**, so interpretation is not based only on papers and not based only on ROI tables.


## Workflow Alignment With Proposal And Slides

This notebook now mirrors the project logic in the proposal:

- **Aim 1**: organize structural MRI outputs from FreeSurfer / FastSurfer / recon-all-clinical
- **Aim 2**: build a structured evidence framework from literature
- **Aim 3**: optionally include DWI, perfusion, cognitive, behavioural, or biomarker data
- **Comparison stage**: compare subject or group features against structured evidence
- **Reporting stage**: produce a traceable summary with explicit caveats and uncertainty

This notebook is designed as an **evidence-informed interpretive support workflow**, not a fully automated diagnostic system.


## Data Sources

This workflow can combine four input layers:

1. **Literature evidence sources**
   - PubMed / PMC URLs
   - local PDF files
   - other text sources describing disease-related imaging patterns
2. **Structural MRI feature table**
   - vertical CSV exported from FreeSurfer / FastSurfer / related pipelines
3. **Normative / control reference table**
   - optional control or age / sex adjusted reference values
4. **Complementary tables**
   - optional DWI, perfusion, cognitive, behavioural, or biomarker data

Generated database artifacts such as `literature_database.sql`, `literature_database.json`, and `literature_database.xlsx` are workflow outputs and should be regenerated locally instead of uploaded.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display, Markdown

project_dir = Path.cwd()
sys.path.append(str(project_dir))

from neurodesk_literature_to_pgsql import (
    analyze_sources,
    build_evidence_framework,
    build_interpretive_report,
    compare_features_to_literature,
    compare_to_normative_reference,
    literature_diseases_for_features,
    literature_features_for_disease,
    normalize_structural_dataframe,
    query_literature,
    query_structural_features,
    summarize_diagnosis_support,
    summarize_structural_features,
)


## Stage 1. Configure Inputs

### Structural MRI as the core input layer

This stage corresponds to **Aim 1** in the proposal.

Configure:

- literature sources for the evidence framework
- structural MRI feature CSV produced by FreeSurfer / FastSurfer / related pipelines
- optional normative reference table
- optional complementary tables
- output directory for generated literature database artifacts


In [ ]:
# LITERATURE SOURCES FOR THE EVIDENCE FRAMEWORK
INPUTS = [
    # 'https://pubmed.ncbi.nlm.nih.gov/21810890/',
    # 'https://pubmed.ncbi.nlm.nih.gov/21325651/',
    # '/home/jovyan/Desktop/paper.pdf',
]

# OUTPUT DIRECTORY FOR LITERATURE DATABASE ARTIFACTS
OUTPUT_DIR = Path('./literature_outputs_workflow_demo')
LITERATURE_JSON_PATH = OUTPUT_DIR / 'literature_database.json'

# STRUCTURAL MRI FEATURE TABLE
# Replace this with your FastSurfer / FreeSurfer vertical ROI feature table
STRUCTURAL_FEATURES_CSV_PATH = Path('/home/jovyan/oasis_selected/ipd_vertical_dataset/roi_vertical_dataset_template.csv')

# OPTIONAL NORMATIVE / CONTROL REFERENCE TABLE
NORMATIVE_CSV_PATH = Path('/home/jovyan/oasis_selected/ipd_vertical_dataset/roi_vertical_dataset_template.csv')
USE_NORMATIVE_REFERENCE = False

# OPTIONAL COMPLEMENTARY TABLES FOR AIM 3
DWI_CSV_PATH = None
PERFUSION_CSV_PATH = None
COGNITIVE_CSV_PATH = None
BIOMARKER_CSV_PATH = None

# AI BACKEND SETTINGS FOR LITERATURE EXTRACTION
AI_BACKEND = 'none'
AI_BASE_URL = 'http://127.0.0.1:11434/v1'
AI_MODEL = 'llama3.1:8b'
AI_API_KEY = None

settings_df = pd.DataFrame([
    {'setting': 'n_literature_inputs', 'value': len(INPUTS)},
    {'setting': 'output_dir', 'value': str(OUTPUT_DIR)},
    {'setting': 'literature_json_path', 'value': str(LITERATURE_JSON_PATH)},
    {'setting': 'structural_features_csv_path', 'value': str(STRUCTURAL_FEATURES_CSV_PATH)},
    {'setting': 'structural_features_exists', 'value': STRUCTURAL_FEATURES_CSV_PATH.exists()},
    {'setting': 'normative_csv_path', 'value': str(NORMATIVE_CSV_PATH)},
    {'setting': 'normative_csv_exists', 'value': NORMATIVE_CSV_PATH.exists()},
    {'setting': 'use_normative_reference', 'value': USE_NORMATIVE_REFERENCE},
    {'setting': 'ai_backend', 'value': AI_BACKEND},
])

display(settings_df)


## Stage 1A. Load Structural MRI Features

### Expected feature table structure

This stage corresponds to the **structured feature outputs** described in the slides.

The feature table should ideally contain columns such as:

- `subject_id`
- `diagnosis`
- `sex`
- `roi_name`
- `imaging_metric`
- `value_numeric`
- `data_level`
- `statistic_type`
- `source_pipeline`
- `hemisphere`

This makes it possible to support both **subject-level** and **group-level** analysis.


In [ ]:
def load_optional_csv(path):
    if path is None:
        return None
    path = Path(path)
    if not path.exists():
        return None
    return pd.read_csv(path)


if not STRUCTURAL_FEATURES_CSV_PATH.exists():
    raise FileNotFoundError(f'Structural feature CSV not found: {STRUCTURAL_FEATURES_CSV_PATH}')

structural_df = pd.read_csv(STRUCTURAL_FEATURES_CSV_PATH)
structural_df = normalize_structural_dataframe(structural_df)

normative_df = None
if USE_NORMATIVE_REFERENCE and NORMATIVE_CSV_PATH.exists():
    normative_df = normalize_structural_dataframe(pd.read_csv(NORMATIVE_CSV_PATH))

display(pd.DataFrame([
    {'item': 'structural_row_count', 'value': len(structural_df)},
    {'item': 'structural_column_count', 'value': len(structural_df.columns)},
    {'item': 'normative_loaded', 'value': normative_df is not None},
]))

display(structural_df.head())


## Stage 1B. Preview The Structural Feature Layer

### Why this stage matters

In proposal terms, this is the reproducible feature-extraction foundation.

Before building comparisons, it is useful to confirm:

- which diagnoses are represented in the structured table
- which ROIs and imaging metrics are available
- whether the table looks like subject-level data or group summaries


In [ ]:
display(Markdown('### Structural feature summary'))
display(summarize_structural_features(structural_df).head(20))

display(Markdown('### Available diagnosis / ROI / metric combinations'))
preview_cols = [c for c in ['subject_id', 'diagnosis', 'diagnosis_normalized', 'roi_name', 'roi_name_normalized', 'imaging_metric', 'imaging_metric_normalized', 'value_numeric', 'source_pipeline'] if c in structural_df.columns]
display(structural_df[preview_cols].head(20))


## Stage 2. Build Or Load The Literature Evidence Framework

### From scattered findings to structured comparison

This stage corresponds to **Aim 2** in the proposal.

The goal is to convert literature into a reusable framework that links:

- disorder
- region
- MRI feature
- reported pattern
- source
- methodological context

Logic used here:

- If `INPUTS` contains sources, run the extraction pipeline.
- If `INPUTS` is empty, load an existing `literature_database.json`.
- Keep the result structured and traceable, rather than treating literature as free text only.


In [ ]:
literature_result = None

if INPUTS:
    literature_result = analyze_sources(
        inputs=INPUTS,
        output_dir=OUTPUT_DIR,
        ai_backend=AI_BACKEND,
        ai_base_url=AI_BASE_URL,
        ai_model=AI_MODEL,
        ai_api_key=AI_API_KEY,
    )
    records = literature_result['records']
else:
    if not LITERATURE_JSON_PATH.exists():
        raise FileNotFoundError(
            f'No INPUTS were provided and no existing literature JSON was found at {LITERATURE_JSON_PATH}'
        )
    records = json.loads(LITERATURE_JSON_PATH.read_text(encoding='utf-8'))

records_df = pd.DataFrame(records)
evidence_df = build_evidence_framework(records)

display(pd.DataFrame([
    {'item': 'literature_record_count', 'value': len(records)},
    {'item': 'used_existing_json', 'value': literature_result is None},
    {'item': 'literature_json_path', 'value': str(LITERATURE_JSON_PATH if literature_result is None else literature_result['json_path'])},
    {'item': 'evidence_framework_rows', 'value': len(evidence_df)},
]))


## Stage 2A. Preview The Evidence Framework

### Make the evidence structure explicit before comparison

This corresponds to the proposal idea that findings should be **reusable, extendable, and traceable**.

Useful preview views include:

- literature rows by diagnosis
- disease-feature summaries
- framework rows organized by disorder, region, and metric


In [ ]:
display(Markdown('### Literature preview columns'))
literature_preview_cols = [
    'title', 'year', 'doi', 'diagnoses', 'rois', 'imaging_metrics',
    'modalities', 'source_type', 'study_design'
]
display(records_df[[c for c in literature_preview_cols if c in records_df.columns]].head(20))

display(Markdown('### Example evidence rows for FTD-spectrum disorders'))
display(query_literature(records, disease='FTD').head(20))

display(Markdown('### Example features reported for Dementia'))
display(literature_features_for_disease(records, 'Dementia').head(20))

display(Markdown('### Reverse search: diseases linked to hippocampus and cortical thickness'))
display(literature_diseases_for_features(records, ['hippocampus', 'cortical_thickness']).head(20))


## Stage 3. Optional Multimodal Extension

### Aim 3 is optional, not a blocker for the core workflow

This stage corresponds to the proposal slide on **complementary information**.

Potential extensions:

- DWI
- perfusion imaging
- cognitive / behavioural data
- blood biomarkers

If these tables are unavailable, the core workflow still remains valid with structural MRI + literature evidence.


In [ ]:
dwi_df = load_optional_csv(DWI_CSV_PATH)
perfusion_df = load_optional_csv(PERFUSION_CSV_PATH)
cognitive_df = load_optional_csv(COGNITIVE_CSV_PATH)
biomarker_df = load_optional_csv(BIOMARKER_CSV_PATH)

multimodal_status_df = pd.DataFrame([
    {'table': 'dwi_df', 'loaded': dwi_df is not None},
    {'table': 'perfusion_df', 'loaded': perfusion_df is not None},
    {'table': 'cognitive_df', 'loaded': cognitive_df is not None},
    {'table': 'biomarker_df', 'loaded': biomarker_df is not None},
])

display(multimodal_status_df)


## Stage 4. Patient-To-Evidence Comparison

### Match structural features against the framework cautiously

This stage corresponds to the comparison logic in the proposal:

- compare patient or group rows with literature-reported patterns
- optionally compare against normative reference data
- keep methodological caveats visible
- treat the output as **interpretive support**, not diagnosis

The Python helper now supports **combined analysis**:

- structural feature signatures
- literature-linked evidence matches
- finer disease differentiation, including FTD subtypes where possible


In [ ]:
display(Markdown('### Example A: Structural rows for Dementia'))
display(query_structural_features(structural_df, diagnosis='Dementia').head(20))

display(Markdown('### Example B: Literature evidence for FTD'))
display(query_literature(records, disease='FTD').head(20))

display(Markdown('### Example C: Match hippocampus ROI features to literature evidence'))
display(compare_features_to_literature(structural_df, evidence_df, roi='hippocampus').head(20))

if USE_NORMATIVE_REFERENCE and normative_df is not None:
    display(Markdown('### Example D: Normative comparison for hippocampus ROI volume'))
    display(compare_to_normative_reference(structural_df, normative_df, roi='hippocampus', metric='roi_volume').head(20))


## Stage 4A. Combined Disease Support Analysis

### Combine literature evidence with structural feature patterns

This is the main change in the updated Python workflow.

Disease support is no longer inferred from literature alone. Instead, the workflow combines:

- ROI / metric patterns from the structural feature table
- disease-linked evidence rows from the literature framework
- subtype-aware signatures, especially for the FTD spectrum

This is intended as cautious analytical support rather than definitive classification.


In [ ]:
display(Markdown('### Combined disease support ranking'))
diagnosis_support_df = summarize_diagnosis_support(structural_df, records)
display(diagnosis_support_df.head(20))


## Stage 5. Structured Interpretive Report

### Output a cautious, traceable summary

This stage corresponds to the **structured report** described in the proposal slides.

The report should:

- restate what structural features were examined
- list evidence-linked matches
- summarize disease support ranking
- flag missing information
- keep uncertainty and methodological caveats explicit


In [ ]:
report_text = build_interpretive_report(
    structural_df=structural_df,
    records=records,
    diagnosis='Dementia',
    roi='hippocampus',
)
display(Markdown(report_text))


## Suggested Presentation Order

A proposal-aligned walkthrough is:

1. Show the configured structural MRI, literature, and optional multimodal inputs.
2. Show the structural feature table and explain the role of FreeSurfer / FastSurfer outputs.
3. Build or load the literature evidence framework.
4. Preview the evidence schema by disorder, region, and metric.
5. Run one comparison example between structural rows and literature evidence.
6. Show the combined disease-support analysis.
7. If available, show a normative comparison.
8. End with the structured interpretive report and caveats.

This sequence now matches the logic presented in the proposal and thesis slides.
